# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [6]:
AviData = pd.read_csv("AviationData.csv", encoding= 'latin1', low_memory= False)
AviData

,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
0,20001218X45444,Accident,SEA87LA080,1948-10-24,"MOOSE CREEK, ID",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,UNK,Cruise,Probable Cause,NaN
1,20001218X45447,Accident,LAX94LA336,1962-07-19,"BRIDGEPORT, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,4.0,0.0,0.0,0.0,UNK,Unknown,Probable Cause,19-09-1996
2,20061025X01555,Accident,NYC07LA005,1974-08-30,"Saltville, VA",United States,36.922223,-81.878056,NaN,NaN,...,Personal,NaN,3.0,NaN,NaN,NaN,IMC,Cruise,Probable Cause,26-02-2007
3,20001218X45448,Accident,LAX96LA321,1977-06-19,"EUREKA, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,IMC,Cruise,Probable Cause,12-09-2000
4,20041105X01764,Accident,CHI79FA064,1979-08-02,"Canton, OH",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,1.0,2.0,NaN,0.0,VMC,Approach,Probable Cause,16-04-1980
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
88884,20221227106491,Accident,ERA23LA093,2022-12-26,"Annapolis, MD",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,0.0,1.0,0.0,0.0,NaN,NaN,NaN,29-12-2022
88885,20221227106494,Accident,ERA23LA095,2022-12-26,"Hampton, NH",United States,NaN,NaN,NaN,NaN,...,NaN,NaN,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN
88886,20221227106497,Accident,WPR23LA075,2022-12-26,"Payson, AZ",United States,341525N,1112021W,PAN,PAYSON,...,Personal,NaN,0.0,0.0,0.0,1.0,VMC,NaN,NaN,27-12-2022
88887,20221227106498,Accident,WPR23LA076,2022-12-26,"Morgan, UT",United States,NaN,NaN,NaN,NaN,...,Personal,MC CESSNA 210N LLC,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN


In [11]:
## Data Cleaning
AviData.info()



<class 'pandas.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  str    
 1   Investigation.Type      88889 non-null  str    
 2   Accident.Number         88889 non-null  str    
 3   Event.Date              88889 non-null  str    
 4   Location                88837 non-null  str    
 5   Country                 88663 non-null  str    
 6   Latitude                34382 non-null  str    
 7   Longitude               34373 non-null  str    
 8   Airport.Code            50132 non-null  str    
 9   Airport.Name            52704 non-null  str    
 10  Injury.Severity         87889 non-null  str    
 11  Aircraft.damage         85695 non-null  str    
 12  Aircraft.Category       32287 non-null  str    
 13  Registration.Number     87507 non-null  str    
 14  Make                    88826 non-null  str    
 

In [15]:
AviData.describe()

,Number.of.Engines,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured
count,82805.000000,77488.000000,76379.000000,76956.000000,82977.000000
mean,1.146585,0.647855,0.279881,0.357061,5.325440
std,0.446510,5.485960,1.544084,2.235625,27.913634
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,0.000000,0.000000,0.000000,1.000000
75%,1.000000,0.000000,0.000000,0.000000,2.000000
max,8.000000,349.000000,161.000000,380.000000,699.000000


### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [19]:
cols = ['Event.Date','Make', 'Model', 'Aircraft.Category',
    'Amateur.Built', 'Number.of.Engines',
    'Total.Fatal.Injuries', 'Total.Serious.Injuries',
    'Total.Minor.Injuries', 'Total.Uninjured',
    'Aircraft.damage', 'Weather.Condition',
    'Broad.phase.of.flight']

AviData[cols].info()

<class 'pandas.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 13 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Date              88889 non-null  str    
 1   Make                    88826 non-null  str    
 2   Model                   88797 non-null  str    
 3   Aircraft.Category       32287 non-null  str    
 4   Amateur.Built           88787 non-null  str    
 5   Number.of.Engines       82805 non-null  float64
 6   Total.Fatal.Injuries    77488 non-null  float64
 7   Total.Serious.Injuries  76379 non-null  float64
 8   Total.Minor.Injuries    76956 non-null  float64
 9   Total.Uninjured         82977 non-null  float64
 10  Aircraft.damage         85695 non-null  str    
 11  Weather.Condition       84397 non-null  str    
 12  Broad.phase.of.flight   61724 non-null  str    
dtypes: float64(5), str(8)
memory usage: 12.8 MB


In [20]:
AviData[cols].isna().sum().sort_values(ascending= False)

Aircraft.Category         56602
Broad.phase.of.flight     27165
Total.Serious.Injuries    12510
Total.Minor.Injuries      11933
Total.Fatal.Injuries      11401
Number.of.Engines          6084
Total.Uninjured            5912
Weather.Condition          4492
Aircraft.damage            3194
Amateur.Built               102
Model                        92
Make                         63
Event.Date                    0
dtype: int64

The Dataset contains key variables related to aircraft identity (Make, Model), Accident Severity (Injuries, Aircraft Damage), and contextual factors (Weather and the phase of flight). Missing values were assessed to determine appropriate cleaning and imputation strategies.

### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [22]:
# Imputation Strats

injury_cols = [
    'Total.Fatal.Injuries',
    'Total.Serious.Injuries',
    'Total.Minor.Injuries',
    'Total.Uninjured'
]

AviData[injury_cols].fillna(0)

,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured
0,2.0,0.0,0.0,0.0
1,4.0,0.0,0.0,0.0
2,3.0,0.0,0.0,0.0
3,2.0,0.0,0.0,0.0
4,1.0,2.0,0.0,0.0
...,...,...,...,...
88884,0.0,1.0,0.0,0.0
88885,0.0,0.0,0.0,0.0
88886,0.0,0.0,0.0,1.0
88887,0.0,0.0,0.0,0.0


The missing values (NaN) in injury-related columns were replaced with zero, as missing values in this dataset indicate no recorded injuries.

**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [23]:
AviData['Aircraft.damage'] = AviData['Aircraft.damage'].fillna("Unknown")

### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [27]:
AviData['Make'].sample(20)

86434       CESSNA
14428        Piper
59691        Piper
54356       Cessna
83299       BOEING
62489       Cessna
75866       CESSNA
22470       Cessna
51983       Cessna
21008         Lake
25129    Schweizer
17312       Cessna
4987        Mooney
74260         BELL
58143       Cessna
20618       Cessna
10947        Piper
580         Cessna
77032       BOEING
86828       CESSNA
Name: Make, dtype: str

In [34]:
AviData['Make'] = AviData['Make'].str.upper().str.strip()
AviData['Make']

0                           STINSON
1                             PIPER
2                            CESSNA
3                          ROCKWELL
4                            CESSNA
                    ...            
88884                         PIPER
88885                      BELLANCA
88886    AMERICAN CHAMPION AIRCRAFT
88887                        CESSNA
88888                         PIPER
Name: Make, Length: 74650, dtype: str

In [67]:
AviData['Make'] = AviData['Make'].replace({
        'MCDONNELL-DOUGLAS' : 'MCDONNELL',
        'AMERICAN CHAMPION AIRCRAFT' : 'AMERICAN CHAMPION',
        'HOWARD AIRCRAFT CORP.' : 'HOWARD',
        'AIRBUS INDUSTRIE' : 'AIRBUS',
        'BOMBARDIER INC' : 'BOMBARDIER',
        'DEHAVILLAND' : 'DE HAVILLAND',
        'GULFSTREAM AMERICAN' : 'GULFSTREAM'
})

AviData['Make']

0                  STINSON
1                    PIPER
2                   CESSNA
3                 ROCKWELL
4                   CESSNA
               ...        
88884                PIPER
88885             BELLANCA
88886    AMERICAN CHAMPION
88887               CESSNA
88888                PIPER
Name: Make, Length: 74627, dtype: str

In [68]:
count = AviData['Make'].value_counts()

valid_makes = count[count >= 50].index

AviData = AviData[AviData['Make'].isin(valid_makes)]

count.head(10)

Make
CESSNA      27145
PIPER       14869
BEECH        5371
BOEING       2738
BELL         2722
MOONEY       1334
ROBINSON     1229
GRUMMAN      1172
BELLANCA     1045
HUGHES        932
Name: count, dtype: int64

## CLEANING THE Make column EXPLANATION

The Make column contained some inconsistencies that had to be addressed before analysis

Issues: 1) Inconsistent Capitalization
        2) Unnecessary spaces
        3) Variations of the same Manufacturer 

Cleaning Steps: 1) Standardize the text to Uppercase
                2) Remove spaces
                3) Simplify manufacturer names

### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [69]:
AviData['Model'].value_counts().head(20)

Model
152          2366
172          1753
172N         1163
PA-28-140     932
150           829
172M          798
172P          689
182           659
180           622
150M          585
PA-18         578
PA-28-180     572
PA-18-150     571
PA-28-161     565
PA-28-181     529
206B          518
737           489
PA-38-112     468
150L          460
G-164A        453
Name: count, dtype: int64

In [70]:
AviData = AviData.dropna(subset = ['Model'])

In [71]:
# Are models unique to each make? Let's find out

model_make = AviData.groupby('Model')['Make'].nunique() 

non_unique_models = model_make[model_make > 1] #determines whether the model appears more than once under multiple manufacturers

non_unique_models.head()

Model
100        4
100-180    2
105        2
109        2
109A       2
Name: Make, dtype: int64

Findings: Same model names used by different manufacturers e.g; 100,105
Also found inconsistent naming e.g; 172, 172N, 172M etc

Therefore, Model is not globally unique

In [72]:
# Let's clean Model just like we did with Make

AviData['Model'] = AviData['Model'].str.upper().str.strip()


In [73]:
# Lets create a Unique Identifier
AviData['AircraftID'] = AviData['Make'] + " " + AviData['Model']
AviData[['Make','Model','AircraftID']]

,Make,Model,AircraftID
0,STINSON,108-3,STINSON 108-3
1,PIPER,PA24-180,PIPER PA24-180
2,CESSNA,172M,CESSNA 172M
3,ROCKWELL,112,ROCKWELL 112
4,CESSNA,501,CESSNA 501
...,...,...,...
88884,PIPER,PA-28-151,PIPER PA-28-151
88885,BELLANCA,7ECA,BELLANCA 7ECA
88886,AMERICAN CHAMPION,8GCBC,AMERICAN CHAMPION 8GCBC
88887,CESSNA,210N,CESSNA 210N


In [74]:
AviData['AircraftID'].value_counts().head(10)

AircraftID
CESSNA 152         2366
CESSNA 172         1753
CESSNA 172N        1163
PIPER PA-28-140     932
CESSNA 150          829
CESSNA 172M         798
CESSNA 172P         689
CESSNA 182          659
CESSNA 180          621
CESSNA 150M         585
Name: count, dtype: int64

## CLEANING MODEL EXPLANATION

The Model column contained missing values and inconsistencies. Records with missing model information were removed, as the model information is essential for aircraft analysis. Moreover, it was revealed that Model is not unique across manufacturers, with identical models appearing under multiple takes. This makes it unreliable to use the Model column alone as an identifier. To address this, a new variable (AircraftID) was created by combining the Make and the Model columns. This ensures that each aircraft type is uniquely identified and boosts accuracy.

### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [75]:
# Lets make a column list to make it easier

columns = ['Engine.Type','Weather.Condition','Number.of.Engines','Purpose.of.flight','Broad.phase.of.flight']

for col in columns:
    print(AviData[col].value_counts(dropna=False).head(10))

Engine.Type
RECIPROCATING      58626
NaN                 5588
TURBO SHAFT         3116
TURBO PROP          2886
TURBO FAN           2220
UNKNOWN             1611
TURBO JET            565
GEARED TURBOFAN       12
LR                     1
UNK                    1
Name: count, dtype: int64
Weather.Condition
VMC        64185
IMC         5601
NaN         3808
UNKNOWN     1033
Name: count, dtype: int64
Number.of.Engines
1.0    57842
2.0    10121
NaN     5007
0.0      795
3.0      450
4.0      411
8.0        1
Name: count, dtype: int64
Purpose.of.flight
PERSONAL              39217
TRAINING              10010
UNKNOWN                6129
NaN                    5442
AERIAL APPLICATION     4293
COMMERCIAL             4231
POSITIONING            1451
OTHER WORK USE         1069
AERIAL OBSERVATION      727
FERRY                   708
Name: count, dtype: int64
Broad.phase.of.flight
NaN            20851
LANDING        13979
TAKEOFF        10550
CRUISE          9044
MANEUVERING     6713
APPROACH      

In [76]:
# Standardizing the values of Engine Type
AviData['Engine.Type'] = AviData['Engine.Type'].str.upper().str.strip()

engine_counts = AviData['Engine.Type'].value_counts()
engine_counts

Engine.Type
RECIPROCATING      58626
TURBO SHAFT         3116
TURBO PROP          2886
TURBO FAN           2220
UNKNOWN             1611
TURBO JET            565
GEARED TURBOFAN       12
LR                     1
UNK                    1
NONE                   1
Name: count, dtype: int64

In [77]:
AviData['Weather.Condition'] = AviData['Weather.Condition'].str.upper().str.strip()

AviData['Weather.Condition'] = AviData['Weather.Condition'].replace({'UNK' : 'UNKNOWN'})

In [78]:
AviData['Number.of.Engines'].value_counts()

Number.of.Engines
1.0    57842
2.0    10121
0.0      795
3.0      450
4.0      411
8.0        1
Name: count, dtype: int64

In [79]:
AviData['Purpose.of.flight'] = AviData['Purpose.of.flight'].str.upper().str.strip()

AviData['Purpose.of.flight'] = AviData['Purpose.of.flight'].replace({'PERSONAL' : 'PERSONAL',
                                                                     'BUSINESS' : 'COMMERCIAL',
                                                                     'EXECUTIVE/CORPORATE' : 'COMMERCIAL',
                                                                     'INSTRUCTIONAL' : 'TRAINING',
                                                                     'FLIGHT TEST' : 'TRAINING',
                                                                    })

purpose_counts = AviData['Purpose.of.flight'].value_counts()
purpose_counts

Purpose.of.flight
PERSONAL                     39217
TRAINING                     10010
UNKNOWN                       6129
AERIAL APPLICATION            4293
COMMERCIAL                    4231
POSITIONING                   1451
OTHER WORK USE                1069
AERIAL OBSERVATION             727
FERRY                          708
PUBLIC AIRCRAFT                666
SKYDIVING                      177
BANNER TOW                      97
EXTERNAL LOAD                   85
PUBLIC AIRCRAFT - FEDERAL       76
PUBLIC AIRCRAFT - STATE         58
PUBLIC AIRCRAFT - LOCAL         52
GLIDER TOW                      41
FIREFIGHTING                    33
AIR RACE SHOW                   30
AIR RACE/SHOW                   18
AIR DROP                        10
PUBS                             3
ASHO                             3
PUBL                             1
Name: count, dtype: int64

In [80]:
AviData['Broad.phase.of.flight'] = AviData['Broad.phase.of.flight'].str.upper().str.strip()

In [81]:
# FINAL CHECKS
for col in columns:
    print(AviData[col].value_counts(dropna= False).head(10))

Engine.Type
RECIPROCATING      58626
NaN                 5588
TURBO SHAFT         3116
TURBO PROP          2886
TURBO FAN           2220
UNKNOWN             1611
TURBO JET            565
GEARED TURBOFAN       12
LR                     1
UNK                    1
Name: count, dtype: int64
Weather.Condition
VMC        64185
IMC         5601
NaN         3808
UNKNOWN     1033
Name: count, dtype: int64
Number.of.Engines
1.0    57842
2.0    10121
NaN     5007
0.0      795
3.0      450
4.0      411
8.0        1
Name: count, dtype: int64
Purpose.of.flight
PERSONAL              39217
TRAINING              10010
UNKNOWN                6129
NaN                    5442
AERIAL APPLICATION     4293
COMMERCIAL             4231
POSITIONING            1451
OTHER WORK USE         1069
AERIAL OBSERVATION      727
FERRY                   708
Name: count, dtype: int64
Broad.phase.of.flight
NaN            20851
LANDING        13979
TAKEOFF        10550
CRUISE          9044
MANEUVERING     6713
APPROACH      

## CLEANING MORE COLUMNS 

Several operational and environmental variables were cleaned to ensure consistency and usefulness. Text-based columns such as engine type, weather conditions, purpose of flight, and phase of flight were standardized to uppercase and stripped of formatting inconsistencies.

Missing values remained untouched as they may contain meaningful information and imputing information could introduce bias.

### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [82]:
missing_data = AviData.isna().mean().sort_values(ascending= False)
missing_data

Airport.Code              0.439827
Airport.Name              0.408793
Broad.phase.of.flight     0.279403
Publication.Date          0.164954
Total.Serious.Injuries    0.138797
Total.Minor.Injuries      0.131762
Total.Fatal.Injuries      0.126844
Engine.Type               0.074879
Purpose.of.flight         0.072923
Report.Status             0.069291
Number.of.Engines         0.067094
Total.Uninjured           0.060648
Weather.Condition         0.051027
Registration.Number       0.015959
Injury.Severity           0.011846
Country                   0.002747
Amateur.Built             0.001179
Location                  0.000657
Event.Id                  0.000000
Investigation.Type        0.000000
Model                     0.000000
Make                      0.000000
Aircraft.damage           0.000000
Event.Date                0.000000
Accident.Number           0.000000
AircraftID                0.000000
dtype: float64

In [83]:
# Lets drop the columns with 60% + of missing values

Threshold = 0.6

drop_col = missing_data[missing_data > Threshold].index
drop_col

Index([], dtype='str')

In [84]:
AviData.shape

(74627, 26)

In [85]:
cols_to_drop = ['Schedule', 'Air.carrier', 'FAR.Description', 'Aircraft.Category',
       'Longitude', 'Latitude']
existing_cols = [col for col in cols_to_drop if col in AviData.columns]
AviData = AviData.drop(columns= existing_cols)
AviData.shape

(74627, 26)

### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [86]:
AviData.to_csv("cleaned_aviation_data.csv", index= False)